In [37]:
import pandas as pd

In [52]:
df_train = pd.read_csv("../clean_data/train_data.csv")

# Preprocess train data
if 'publish_time_utc' in df_train.columns:
    df_train.drop(columns=['publish_time_local', 'publish_time_utc'], inplace=True)

# Add 'solar_' prefix to columns by position 3 through 8 (inclusive)
start_idx, end_idx = 2, 7
cols = list(df_train.columns)
for i in range(start_idx, end_idx + 1):
    if i < len(cols):
        col = cols[i]
        if not col.startswith('solar_'):
            cols[i] = 'solar_' + col
df_train.columns = cols

if 'solar_gen_system_wide_x' in df_train.columns:    # Remove trailing '_x' if present
    df_train.rename(columns={'solar_gen_system_wide_x': 'solar_gen_system_wide'}, inplace=True)

# Add 'wind_' prefix to columns by position 9 through 12 (inclusive)
start_idx, end_idx = 8, 11
cols = list(df_train.columns)
for i in range(start_idx, end_idx + 1):
    if i < len(cols):
        col = cols[i]
        if not col.startswith('wind_'):
            cols[i] = 'wind_' + col
df_train.columns = cols

if 'wind_gen_system_wide_y' in df_train.columns:   # Remove trailing '_y' if present
    df_train.rename(columns={'wind_gen_system_wide_y': 'wind_gen_system_wide'}, inplace=True)

# Choose hub to predict
hubs = ['HB_BUSAVG', 'HB_HOUSTON', 'HB_HUBAVG', 'HB_NORTH', 'HB_PAN', 'HB_SOUTH', 'HB_WEST']
hub_to_predict = 'HB_NORTH'

# Keep only the chosen hub column in df_train (and df_test if available).
#  - Drops other hub columns listed in `hubs`.
existing_hubs_train = [h for h in hubs if h in df_train.columns]
# Drop other hubs from df_train
drop_hubs_train = [h for h in existing_hubs_train if h != hub_to_predict]
if drop_hubs_train:
    df_train.drop(columns=drop_hubs_train, inplace=True)

# Show result
df_train.iloc[:5, 8:]
df_train


,interval_start_local,load,solar_gen_system_wide,solar_gen_centerwest,solar_gen_northwest,solar_gen_fareast,solar_gen_southeast,solar_gen_centereast,wind_gen_system_wide,wind_gen_lz_south_houston,...,TEMP_C,TEMP_qc,DEW_C,DEW_qc,SLP_hPa,SLP_qc,WIND_DIR_deg,WIND_DIR_qc,WIND_SPD_ms,WIND_SPD_qc
0,2023-01-01 00:00:00,34969.250000,0.45,0.01,0.00,0.36,0.00,0.07,21752.91,4318.38,...,17.75,3.0,6.70,3.0,1007.75,3.0,185.0,3.0,4.1,3.0
1,2023-01-01 01:00:00,35573.500000,0.46,0.01,0.00,0.37,0.00,0.07,21569.51,3685.25,...,16.70,5.0,7.20,5.0,1007.90,5.0,180.0,5.0,4.6,5.0
2,2023-01-01 02:00:00,36279.750000,0.45,0.01,0.00,0.36,0.00,0.07,21035.48,3544.24,...,16.10,5.0,7.20,5.0,1008.70,5.0,170.0,5.0,3.1,5.0
3,2023-01-01 03:00:00,36765.833333,0.46,0.01,0.00,0.37,0.00,0.07,20595.37,3571.74,...,15.85,3.0,7.20,3.0,1008.80,3.0,175.0,3.0,3.6,3.0
4,2023-01-01 04:00:00,37049.916667,0.45,0.01,0.00,0.36,0.00,0.07,20387.59,3295.90,...,15.00,5.0,7.20,5.0,1009.40,5.0,190.0,5.0,4.6,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17509,2024-12-30 13:00:00,49497.333333,10308.84,470.95,962.55,4496.30,1444.14,1412.20,21834.62,2976.84,...,11.10,5.0,7.20,5.0,1005.10,5.0,180.0,5.0,5.7,5.0
17510,2024-12-30 14:00:00,48549.583333,12919.12,1141.19,1005.58,5351.85,1986.38,1583.28,19823.11,2560.70,...,12.20,5.0,7.20,5.0,1004.50,5.0,190.0,5.0,8.2,5.0
17511,2024-12-30 15:00:00,47718.000000,13880.78,1272.04,1031.21,5727.15,2126.71,1648.66,17393.12,2072.08,...,13.60,3.0,6.95,3.0,1004.35,3.0,195.0,3.0,7.7,3.0
17512,2024-12-30 16:00:00,46065.583333,10174.55,1162.50,762.87,3241.28,1440.12,1036.28,14325.53,1804.63,...,20.60,5.0,3.30,5.0,1003.40,5.0,220.0,5.0,8.2,5.0


In [47]:
df_test = pd.read_csv("../clean_data/test_data_baseline.csv")
df_test.drop(columns=['interval_end_local', 'publish_time_local', 'publish_time_utc', 'interval_start_utc', 'interval_end_utc'], inplace=True)
df_test

,interval_start_local,coast,east,far_west,north,north_central,south_central,southern,west,system_total,...,TEMP_C,TEMP_qc,DEW_C,DEW_qc,SLP_hPa,SLP_qc,WIND_DIR_deg,WIND_DIR_qc,WIND_SPD_ms,WIND_SPD_qc
0,2025-01-01 00:00:00,10238.69,1634.45,7645.03,1605.73,12202.98,6384.71,3293.52,1101.77,44106.88,...,6.4,3.0,1.4,3.0,1023.5,3.0,20.0,3.0,4.90,3.0
1,2025-01-01 01:00:00,10121.10,1600.29,7593.90,1624.66,12120.37,6436.58,3259.80,1059.43,43816.13,...,5.6,5.0,1.1,5.0,1024.4,5.0,10.0,5.0,4.60,5.0
2,2025-01-01 02:00:00,10140.17,1576.62,7581.00,1616.56,11994.06,6466.56,3200.95,1058.03,43633.94,...,5.0,5.0,0.6,5.0,1025.0,5.0,20.0,5.0,4.10,5.0
3,2025-01-01 03:00:00,10171.89,1598.36,7403.48,1660.17,12178.42,6425.03,2985.95,1097.81,43521.12,...,4.7,3.0,0.6,3.0,1025.3,3.0,25.0,3.0,4.35,3.0
4,2025-01-01 04:00:00,9992.42,1628.75,7358.83,1640.86,12308.32,6385.58,2964.56,1114.53,43393.86,...,3.9,5.0,0.0,5.0,1026.0,5.0,20.0,5.0,4.10,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5825,2025-08-31 19:00:00,15127.47,1946.72,7831.26,2380.52,17613.82,10899.10,6106.69,1668.67,63574.24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5826,2025-08-31 20:00:00,15143.34,1907.08,7864.67,2361.22,17184.47,10496.14,6034.67,1685.25,62676.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5827,2025-08-31 21:00:00,15002.67,1838.35,7927.39,2322.24,16691.57,10133.57,5746.13,1663.76,61325.69,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5828,2025-08-31 22:00:00,14570.61,1765.04,7888.14,2244.99,15970.11,9704.50,5383.60,1646.29,59173.28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Feature Engineering
# Lags
lag_amount = [1, 2, 4, 8, 12, 24, 48, 72]
lag_features = ['load', 'gen_syst']

# Define column subsets
# Wind columns

def time_features(df):
    df['day_of_week'] = pd.to_datetime(df['interval_start_local']).dt.dayofweek
    df['weekend'] = df['day_of_week'].apply(lambda x: 1 if x in [5, 6] else 0)

def weather_features(df):
    df['TEMP_F'] = df['TEMP_C'] * 9/5 + 32
    df['degree_days'] = abs(df['TEMP_F'] - 65)


In [ ]:
#time_features(df_train)
#weather_features(df_train)



,interval_start_local,load,gen_system_wide_x,gen_centerwest,gen_northwest,gen_fareast,gen_southeast,gen_centereast,gen_system_wide_y,gen_lz_south_houston,...,TEMP_C,TEMP_qc,DEW_C,DEW_qc,SLP_hPa,SLP_qc,WIND_DIR_deg,WIND_DIR_qc,WIND_SPD_ms,WIND_SPD_qc
0,2023-01-01 00:00:00,34969.250000,0.45,0.01,0.00,0.36,0.00,0.07,21752.91,4318.38,...,17.75,3.0,6.70,3.0,1007.75,3.0,185.0,3.0,4.1,3.0
1,2023-01-01 01:00:00,35573.500000,0.46,0.01,0.00,0.37,0.00,0.07,21569.51,3685.25,...,16.70,5.0,7.20,5.0,1007.90,5.0,180.0,5.0,4.6,5.0
2,2023-01-01 02:00:00,36279.750000,0.45,0.01,0.00,0.36,0.00,0.07,21035.48,3544.24,...,16.10,5.0,7.20,5.0,1008.70,5.0,170.0,5.0,3.1,5.0
3,2023-01-01 03:00:00,36765.833333,0.46,0.01,0.00,0.37,0.00,0.07,20595.37,3571.74,...,15.85,3.0,7.20,3.0,1008.80,3.0,175.0,3.0,3.6,3.0
4,2023-01-01 04:00:00,37049.916667,0.45,0.01,0.00,0.36,0.00,0.07,20387.59,3295.90,...,15.00,5.0,7.20,5.0,1009.40,5.0,190.0,5.0,4.6,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17509,2024-12-30 13:00:00,49497.333333,10308.84,470.95,962.55,4496.30,1444.14,1412.20,21834.62,2976.84,...,11.10,5.0,7.20,5.0,1005.10,5.0,180.0,5.0,5.7,5.0
17510,2024-12-30 14:00:00,48549.583333,12919.12,1141.19,1005.58,5351.85,1986.38,1583.28,19823.11,2560.70,...,12.20,5.0,7.20,5.0,1004.50,5.0,190.0,5.0,8.2,5.0
17511,2024-12-30 15:00:00,47718.000000,13880.78,1272.04,1031.21,5727.15,2126.71,1648.66,17393.12,2072.08,...,13.60,3.0,6.95,3.0,1004.35,3.0,195.0,3.0,7.7,3.0
17512,2024-12-30 16:00:00,46065.583333,10174.55,1162.50,762.87,3241.28,1440.12,1036.28,14325.53,1804.63,...,20.60,5.0,3.30,5.0,1003.40,5.0,220.0,5.0,8.2,5.0
